# Regulation PDF audit
Read-only inspection of the source PDFs. This measures extraction readiness, not product applicability or legal completeness. Outputs are written under `pipeline_output/audit/`.

Run top to bottom. Install dependencies in the selected notebook kernel once. The optional Docling pilot is deliberately limited because model downloads and conversion can be expensive.

In [ ]:
%pip install pypdf pymupdf pandas python-dotenv


In [ ]:
from pathlib import Path
import json, hashlib, re, time, csv, traceback
# Locate the repository whether the kernel starts at its root or notebook folder.
PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / ".git").exists()
                and (p / "pdf-audit-and-extract").is_dir()), None)
if PROJECT is None:
    raise RuntimeError("Start the notebook kernel inside the certivity-hack repository.")
from dotenv import load_dotenv
load_dotenv(PROJECT / ".env", override=False)
ROOT = PROJECT
if not (ROOT / "industrial-ai-hackathon").exists():
    ROOT = Path("/Users/mkhungar/Desktop/Exercises/Hacks/Certivity")
PACKAGE = ROOT / "industrial-ai-hackathon"
PDF_ROOT = PACKAGE / "regulations" / "pdf"
OUTPUT = PROJECT / "pipeline_output"
OUTPUT.mkdir(exist_ok=True)
def digest(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()
def write_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False))
    temporary.replace(path)
index = json.loads((PACKAGE / "regulations/index.json").read_text())
metadata = {}
for regulation in index:
    slug = "_".join(regulation["regulation"].replace("/", " ").split())
    for version in regulation["versions"]:
        for file in version["files"]:
            metadata[f"{slug}/{version['role']}/{file['name']}"] = {
                "regulation": regulation["regulation"], "title": regulation["title"],
                "version": version["version_label"], "role": version["role"],
                "expected_sha256": file["sha256"]}
pdfs = sorted(PDF_ROOT.rglob("*.pdf"))
assert pdfs, f"No PDFs found in {PDF_ROOT}"
print(f"Found {len(pdfs)} PDFs; outputs: {OUTPUT}")


In [ ]:
import fitz
from pypdf import PdfReader
AUDIT = OUTPUT / "audit"
AUDIT.mkdir(exist_ok=True)
rows, page_rows = [], []
for path in pdfs:
    started = time.perf_counter()
    key = path.relative_to(PDF_ROOT).as_posix()
    row = {"file": key, **metadata.get(key, {}), "bytes": path.stat().st_size}
    try:
        row["sha256"] = digest(path)
        row["checksum_ok"] = row["sha256"] == row.get("expected_sha256")
        doc = fitz.open(path)
        texts = []
        for i, page in enumerate(doc):
            text = page.get_text("text")
            texts.append(text)
            # Independent text extractor checks broad extraction disagreement.
            page_rows.append({"file": key, "page": i+1, "characters": len(text.strip()),
                "images": len(page.get_images()), "vector_paths": len(page.get_drawings())})
        alternate = PdfReader(path)
        alt_chars = sum(len((p.extract_text() or '').strip()) for p in alternate.pages)
        text = "\n".join(texts)
        row.update(pages=len(doc), characters=len(text),
            low_text_pages=sum(len(t.strip()) < 80 for t in texts),
            numbered_lines=len(re.findall(r"(?m)^\s*\d+(?:\.\d+)*\.?\s+",text)),
            annex_mentions=len(re.findall(r"(?im)^\s*Annex\s+\d+",text)),
            table_mentions=len(re.findall(r"(?im)^\s*Table\s+\w+",text)),
            figure_mentions=len(re.findall(r"(?im)^\s*Figure\s+\w+",text)),
            extractor_character_ratio=round(alt_chars/max(1,len(text)),3))
        flags=[]
        if not row['checksum_ok']: flags.append('checksum mismatch or unknown source')
        if row['low_text_pages']: flags.append('low-text pages: inspect figures/OCR need')
        if row['numbered_lines']==0: flags.append('numbered hierarchy not detected')
        if row['table_mentions'] or row['figure_mentions']: flags.append('table/figure fidelity requires review')
        if not .75 <= row['extractor_character_ratio'] <= 1.25: flags.append('extractors disagree')
        row['status']='needs_review' if flags else 'ready_for_pilot'
        row['flags']='; '.join(flags)
        # Save representative first, table and low-text pages for visual review.
        chosen={0}
        chosen.update(next((i for i,t in enumerate(texts) if re.search(r"(?im)^Table\s",t)),0) for _ in [0])
        chosen.update(i for i,t in enumerate(texts) if len(t.strip())<80)
        sample_dir=AUDIT/'review_samples'/Path(key).with_suffix('')
        sample_dir.mkdir(parents=True,exist_ok=True)
        for i in sorted(chosen)[:4]:
            doc[i].get_pixmap(matrix=fitz.Matrix(1.2,1.2)).save(str(sample_dir/f'page-{i+1}.png'))
            (sample_dir/f'page-{i+1}.txt').write_text(texts[i])
        doc.close()
    except Exception as exc:
        row.update(status='failed',error=f'{type(exc).__name__}: {exc}')
    row['seconds']=round(time.perf_counter()-started,2)
    rows.append(row)
write_json(AUDIT/'inventory.json',rows)
write_json(AUDIT/'pages.json',page_rows)
missing=sorted(set(metadata)-{p.relative_to(PDF_ROOT).as_posix() for p in pdfs})
write_json(AUDIT/'missing-downloads.json',missing)
import pandas as pd
report=pd.DataFrame(rows)
report.to_csv(AUDIT/'extraction-report.csv',index=False)
display(report.groupby('status').size())
display(report[['file','pages','status','flags','seconds']])
print('Missing indexed PDFs:',len(missing))


## Optional Markdown conversion pilot
Select a few different regulation families and versions. Review the Markdown against saved page images before scaling. Heuristic readiness is not proof of accurate tables or complete obligations.

In [ ]:
# Uncomment in this kernel to enable the optional pilot.
# %pip install docling


In [ ]:
RUN_DOCLING_PILOT = False
PILOT_FILES = ["UN_R150/base/UN_R150_base.pdf", "UN_R43/base/UN_R43_base.pdf", "UN_R162/base/UN_R162_base.pdf"]
if RUN_DOCLING_PILOT:
    from docling.document_converter import DocumentConverter
    converter=DocumentConverter()
    pilot=[]
    for key in PILOT_FILES:
        started=time.perf_counter()
        try:
            result=converter.convert(PDF_ROOT/key)
            target=AUDIT/'pilot'/Path(key).with_suffix('')
            target.mkdir(parents=True,exist_ok=True)
            result.document.save_as_json(target/'document.json')
            result.document.save_as_markdown(target/'document.md')
            pilot.append({'file':key,'seconds':time.perf_counter()-started,'status':str(result.status)})
        except Exception as exc: pilot.append({'file':key,'error':str(exc)})
    write_json(AUDIT/'pilot-report.json',pilot)
